In [ ]:
#!pip install scipy

In [ ]:
#!pip install pyDOE

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from pydoe import lhs
from tqdm.notebook import tqdm #진행률
import matplotlib.pyplot as plt

# Domain setting

In [3]:
tmin = 0.0; tmax = np.pi/2 #시간 범위
xmin = -5.0; xmax = 5.0  #공간 범위

lb = np.array([tmin, xmin]) #[0.  -5.] , 벡터((2,)
ub = np.array([tmax, xmax]) #[1.57079633 5.], 벡터((2,)

#초기조건 (t=0)
N_0 = 50
t_0 = np.zeros([N_0, 1])
x_0 = np.random.uniform(xmin, xmax, (N_0, 1))
tx_0 = np.hstack([t_0, x_0]) # (50,2) 행렬

u_0 = 2/np.cosh(x_0) # h(0,x)= 2*sech(x) = 실수부 
v_0 = np.zeros((N_0, 1)) # 0 = 허수부  
uv_0 = np.hstack([u_0, v_0])

#경계조건 ((t,-5) vs (t,5))
N_b = 50
t_b = np.random.uniform(tmin, tmax, (N_b, 1))
tx_lb = np.hstack([t_b, np.full((N_b,1), xmin)])
tx_ub = np.hstack([t_b, np.full((N_b,1), xmax)])

#f에서 사용할 collocation 
N_f =20000
tx_f = lb + (ub - lb)*lhs(2, N_f) #[0,1]에서 (20000,2)행렬 만든 후 범위 조정하여 실제 도메인 범위로 늘려주기

#tensor로 변환
tx_0 = torch.tensor(tx_0, dtype=torch.float32)
uv_0 = torch.tensor(uv_0, dtype=torch.float32)
tx_lb = torch.tensor(tx_lb, requires_grad=True, dtype=torch.float32)
tx_ub = torch.tensor(tx_ub, requires_grad=True, dtype=torch.float32)
tx_f = torch.tensor(tx_f, requires_grad=True, dtype=torch.float32)

# MLP 만들기

In [4]:
class PINN(nn.Module):
    def __init__(self) :                    
        super(PINN, self).__init__()
        self.linear1 = nn.Linear(2, 100) #입력: (x,t), 한 layer당 100뉴런 => (weight:(100,2)행렬 생성, bias:(100,)벡터 생성
        nn.init.xavier_normal_(self.linear1.weight) # w의 쓰레기값 대신 정규분포(mean=0, std=sqrt(2/2+100) 내의 값으로 변환
        
        self.linear2 = nn.Linear(100, 100)
        nn.init.xavier_normal_(self.linear2.weight)

        self.linear3 = nn.Linear(100, 100)
        nn.init.xavier_normal_(self.linear3.weight)

        self.linear4 = nn.Linear(100, 100)
        nn.init.xavier_normal_(self.linear4.weight)

        self.linear5 = nn.Linear(100, 2)
        nn.init.xavier_normal_(self.linear5.weight)

        self.act = nn.Tanh()                         # Nonlinear activation function

    def forward(self, x) :
        h1 = self.act(self.linear1(x))
        h2 = self.act(self.linear2(h1))
        h3 = self.act(self.linear3(h2))
        h4 = self.act(self.linear4(h3))
        o = self.linear5(h4)  #output에는 activation funtion 없음.
        return o

# loss function 만들기

In [5]:
def loss_0(model_h, tx_0, uv_0):
    
    pred_0 = model_h(tx_0)
    mse_0 = ((pred_0 - uv_0)**2).mean()
    
    return mse_0

In [6]:
def loss_b(model_h, tx_lb, tx_ub):
    pred_lb = model_h(tx_lb)
    pred_ub = model_h(tx_ub)


    u_pred_lb = pred_lb[:, 0:1]
    u_x_lb = torch.autograd.grad(outputs = u_pred_lb.sum(), inputs = tx_lb,
                                create_graph=True)[0][:, 1:2]
    
    u_pred_ub = pred_ub[:, 0:1]
    u_x_ub = torch.autograd.grad(outputs = u_pred_ub.sum(), inputs = tx_ub,
                                create_graph=True)[0][:, 1:2]
    
    v_pred_lb = pred_lb[:, 1:2]
    v_x_lb = torch.autograd.grad(outputs = v_pred_lb.sum(), inputs = tx_lb,
                                create_graph=True)[0][:, 1:2]
    
    v_pred_ub = pred_ub[:, 1:2]
    v_x_ub = torch.autograd.grad(outputs = v_pred_ub.sum(), inputs = tx_ub,
                                create_graph=True)[0][:, 1:2]

  
    gap_u = ((u_pred_lb - u_pred_ub)**2)
    gap_v = ((v_pred_lb - v_pred_ub)**2)
    gap = torch.cat([gap_u, gap_v], dim=1).mean()

    grad_gap_u = (u_x_lb - u_x_ub)**2
    grad_gap_v = (v_x_lb - v_x_ub)**2
    grad_gap = torch.cat([grad_gap_u, grad_gap_v], dim=1).mean()

    mse_b =  gap + grad_gap
    
    return mse_b
    

In [7]:
def loss_f(model_h, tx_f):
    output = model_h(tx_f) # [[u(t,x), v(t,x)]]

    #실수부
    u = output[:, 0:1]
    u_tx = torch.autograd.grad(outputs=u.sum(), inputs=tx_f,
                               create_graph=True)[0]
    u_t = u_tx[:, 0:1]
    u_x = u_tx[:, 1:2]
    
    u_xx = torch.autograd.grad(outputs=u_x.sum(), inputs=tx_f,
                               create_graph=True)[0][:,1:2]
    
    
    #허수부
    v = output[:, 1:2]
    v_tx = torch.autograd.grad(outputs=v.sum(), inputs=tx_f,
                               create_graph=True)[0]
    v_t = v_tx[:, 0:1]
    v_x = v_tx[:, 1:2]
    
    v_xx = torch.autograd.grad(outputs=v_x.sum(), inputs=tx_f,
                               create_graph=True)[0][:,1:2]
    
    
    #f 정의 : ih_t + 0.5h_xx + |h|²h = 0에 h = u + iv 대입
    f_u = -v_t + 0.5*u_xx + (u**2 + v**2)*u #실수부
    f_v = u_t + 0.5*v_xx + (u**2 + v**2)*v #허수부

    mse_f = (f_u**2 + f_v**2).mean()

    return mse_f

# train setting

In [8]:
model_h = PINN()
optimizer = torch.optim.LBFGS(
    model_h.parameters(),
    lr=1,
    max_iter=1,
    max_eval=5,
    history_size=50,        # maxcor
    tolerance_change=1.0 * np.finfo(float).eps,
    line_search_fn="strong_wolfe"
)
epochs = 10000

# train

In [9]:
model_h.train()

for epoch in tqdm(range(epochs)):
    def closure():
        optimizer.zero_grad()
        mse_0 = loss_0(model_h, tx_0, uv_0)
        mse_b = loss_b(model_h, tx_lb, tx_ub)
        mse_f = loss_f(model_h, tx_f)
        mse = mse_0 + mse_b + mse_f
        mse.backward()

        return mse

    optimizer.step(closure)

  0%|          | 0/10000 [00:00<?, ?it/s]

# 오차 확인

## setting

In [10]:
from scipy import io
data = io.loadmat('data/NLS.mat')

In [13]:
data.keys()

dict_keys(['__header__', '__version__', '__globals__', 'tt', 'uu', 'x'])

In [45]:
t = data['tt'].reshape(-1,1)
x = data['x'].reshape(-1,1)
exact = data['uu'].T #정답지
u_exact = exact.real #정답지 실수부
v_exact = exact.imag #정답지 허수부

print(f't:{t}\n t.shape:{t.shape}')
print('======================================================')
print(f'x:{x}, x.shape:{x.shape}')
print('======================================================')
print(f'exact.shape:{exact.shape}')
print('======================================================')
print(f'u_exact.shape:{u_exact.shape}')
print('======================================================')
print(f'v_exact.shape:{v_exact.shape}')

t:[[0.        ]
 [0.00785398]
 [0.01570796]
 [0.02356194]
 [0.03141593]
 [0.03926991]
 [0.04712389]
 [0.05497787]
 [0.06283185]
 [0.07068583]
 [0.07853982]
 [0.0863938 ]
 [0.09424778]
 [0.10210176]
 [0.10995574]
 [0.11780972]
 [0.12566371]
 [0.13351769]
 [0.14137167]
 [0.14922565]
 [0.15707963]
 [0.16493361]
 [0.1727876 ]
 [0.18064158]
 [0.18849556]
 [0.19634954]
 [0.20420352]
 [0.2120575 ]
 [0.21991149]
 [0.22776547]
 [0.23561945]
 [0.24347343]
 [0.25132741]
 [0.25918139]
 [0.26703538]
 [0.27488936]
 [0.28274334]
 [0.29059732]
 [0.2984513 ]
 [0.30630528]
 [0.31415927]
 [0.32201325]
 [0.32986723]
 [0.33772121]
 [0.34557519]
 [0.35342917]
 [0.36128316]
 [0.36913714]
 [0.37699112]
 [0.3848451 ]
 [0.39269908]
 [0.40055306]
 [0.40840704]
 [0.41626103]
 [0.42411501]
 [0.43196899]
 [0.43982297]
 [0.44767695]
 [0.45553093]
 [0.46338492]
 [0.4712389 ]
 [0.47909288]
 [0.48694686]
 [0.49480084]
 [0.50265482]
 [0.51050881]
 [0.51836279]
 [0.52621677]
 [0.53407075]
 [0.54192473]
 [0.54977871]
 [0.

In [38]:
T_grid, X_grid = np.meshgrid(t,x, indexing='ij')

print(f'T_grid:{T_grid}\n T_grid.shape:{T_grid.shape}')
print('======================================================')
print(f'X_grid:{X_grid}\n X_grid.shape:{X_grid.shape}')
print('======================================================')

tx_test = np.hstack([T_grid.reshape(-1,1), X_grid.reshape(-1,1)])
print(f'tx_test:{tx_test}\n tx_test.shape:{tx_test.shape}')
print('======================================================')

tx_test = torch.tensor(tx_test, dtype=torch.float32)
print(f'tensor_tx_test:{tx_test}\n tensor_tx_test.shape:{tx_test.shape}')
print('======================================================')

T_grid:[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.00785398 0.00785398 0.00785398 ... 0.00785398 0.00785398 0.00785398]
 [0.01570796 0.01570796 0.01570796 ... 0.01570796 0.01570796 0.01570796]
 ...
 [1.55508836 1.55508836 1.55508836 ... 1.55508836 1.55508836 1.55508836]
 [1.56294235 1.56294235 1.56294235 ... 1.56294235 1.56294235 1.56294235]
 [1.57079633 1.57079633 1.57079633 ... 1.57079633 1.57079633 1.57079633]]
 T_grid.shape:(201, 256)
X_grid:[[-5.        -4.9609375 -4.921875  ...  4.8828125  4.921875   4.9609375]
 [-5.        -4.9609375 -4.921875  ...  4.8828125  4.921875   4.9609375]
 [-5.        -4.9609375 -4.921875  ...  4.8828125  4.921875   4.9609375]
 ...
 [-5.        -4.9609375 -4.921875  ...  4.8828125  4.921875   4.9609375]
 [-5.        -4.9609375 -4.921875  ...  4.8828125  4.921875   4.9609375]
 [-5.        -4.9609375 -4.921875  ...  4.8828125  4.921875   4.9609375]]
 X_grid.shape:(201, 256)
tx_test:[[ 0.         -5.        ]
 [ 0.        

## 확인

In [57]:
model_h.eval()

with torch.no_grad():
    pred = model_h(tx_test).detach().numpy()

u_pred = pred[:,0:1].reshape(T_grid.shape)
v_pred = pred[:,1:2].reshape(T_grid.shape)

h_pred = np.sqrt(u_pred**2 + v_pred**2)
h_exact = np.sqrt(u_exact**2 + v_exact**2)

L2_error = np.linalg.norm(h_pred - h_exact) / np.linalg.norm(h_exact)
print(f'L2_error: {L2_error:.2e}')

L2_error: 2.00e-03
